# 30 — M2 low-cutoff 4D SU(2) armillary

## 判定（2026-07-20 UTC）

**M1 は M2 実装へ進むための前提として受理します。M2 を実行してよいです。**

- 対象 run: `M1-20260719T235423Z-a7cacde2ead9`
- M1 状態: `M1_COMPLETE`、checkpoint 14、全 acceptance gate 成功
- 独立照合: 12/12 成功、proof artifact 6 phase、heuristic result なし
- 許可する範囲: `j2_max=1` の M2 low-cutoff 4D link-star/armillary 実装と検証
- 状態: **`NOT_CERTIFIED` のまま**
- 禁止する解釈: M1/M2 だけから 4D RG、mass gap、熱力学極限、連続極限、または `CERTIFIED` を主張しない

機械可読な判定と固定 SHA-256 は `audit/m1_accepted_parent.json` にあります。以下のセルは毎回それを再検証し、一つでも変化・欠落があれば fail closed で停止します。


## 実行前に読むこと

永続保存先を確認してから、環境変数を設定してください。

```bash
export VALIDATED_RG_PERSIST_ROOT=/storage/validated_4d_su2_rg
export VALIDATED_RG_PERSIST_ACK=I_CONFIRM_THIS_PATH_IS_PERSISTENT
# 既存 M2 を再開するときだけ:
export VALIDATED_RG_M2_RUN_ID=M2-...
```

この workflow は 15 分ごと、全 phase 境界、高コスト基底変換の前後に atomic checkpoint を保存します。5 時間以降は長い処理を開始せず、5 時間20分までに最終保存を開始し、5 時間30分までに返ります。

### 人間が別セッションから再開する方法

1. 同じ永続 volume を mount する。
2. 上の 2 個の永続保存環境変数を設定する。
3. 前回表示された `VALIDATED_RG_M2_RUN_ID` を設定する。
4. fresh kernel でこのノートブックを上から実行する。
5. source・notebook・config・M1 親・checkpoint の hash が一致したときだけ、自動的に最新の有効 checkpoint から続行する。最新が破損していれば一つ前へ戻る。

GitHub push は SSH 鍵登録後に `git push -u origin main` で行います。秘密鍵や token をノートブックへ書かないでください。


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'AGENTS.md').is_file():
    raise RuntimeError(f'Project root を特定できません: {PROJECT_ROOT}')
sys.path.insert(0, str(PROJECT_ROOT))

missing = [
    requirement for requirement, module in (('pytest>=8', 'pytest'), ('sympy>=1.12', 'sympy'))
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import pytest
import sympy
print('project:', PROJECT_ROOT)
print('pytest:', pytest.__version__)
print('sympy:', sympy.__version__)


In [ ]:
import json

from src.m2_config import M2Config
from src.m2_parent import verify_accepted_m1_parent
from src.common import read_json

M2_CONFIG = M2Config()
PARENT_HASHES = verify_accepted_m1_parent(PROJECT_ROOT, M2_CONFIG)
M1_AUDIT = read_json(PROJECT_ROOT / M2_CONFIG.parent_audit_path)
HUMAN_DECISION = {
    '判定': M1_AUDIT['human_readable_decision_ja'],
    'M1 run': M1_AUDIT['accepted_run_id'],
    'M1 phase': M1_AUDIT['accepted_phase'],
    'M2開始': M1_AUDIT['decision'],
    'certification': M1_AUDIT['certification_status'],
    'report_sha256': PARENT_HASHES['m1_report_sha256'],
    'checkpoint_hashes_sha256': PARENT_HASHES['parent_checkpoint_hash_manifest_sha256'],
}
print(json.dumps(HUMAN_DECISION, ensure_ascii=False, indent=2))
assert M1_AUDIT['decision'] == 'ACCEPT_M1_FOR_M2_IMPLEMENTATION'
assert M1_AUDIT['certification_status'] == 'NOT_CERTIFIED'


## M2 の厳密比較

6 個の plaquette leg を持つ固定 4D link star、`j2=2j ∈ {0,1}` の全 64 sector を対象にします。

- dense reference: 全 SU(2) 生成子の同時零空間から Haar singlet projector を厳密に構成
- armillary: Condon–Shortley CG 位相、left-associated fusion tree、向き `(+,-,+,-,+,-)`、双対写像 `C|j,m> = (-1)^(j-m)|j,-m>` を固定
- 判定: SymPy の exact symbolic matrix が要素ごとに一致した場合のみ PASS
- float64 tensor shard: restart 診断専用で、証明や bound には使わない


In [ ]:
import json
import os
import subprocess
import sys
import time

test_started = time.monotonic()
m0_m1_files = [
    'tests/test_m0.py',
    'tests/test_m1_exact_arithmetic.py',
    'tests/test_m1_coefficients.py',
    'tests/test_m1_tails.py',
    'tests/test_m1_exact_2d_rg.py',
    'tests/test_m1_independent_verifier.py',
    'tests/test_m1_restart.py',
    'tests/test_m1_fail_closed.py',
]
m2_files = [
    'tests/test_m2_fusion.py',
    'tests/test_armillary_equivalence.py',
    'tests/test_m2_restart.py',
]

def run_required(paths):
    completed = subprocess.run(
        [sys.executable, '-m', 'pytest', '-q', *paths, '-m', 'not gpu'],
        cwd=PROJECT_ROOT, check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(f'Required pytest suite failed: {paths}, exit={completed.returncode}')

run_required(m0_m1_files)
run_required(m2_files)

cuda_available = False
try:
    import torch
    cuda_available = torch.cuda.is_available()
except ImportError:
    pass
gpu_status = 'NOT_RUN_NO_CUDA'
if cuda_available:
    completed = subprocess.run(
        [sys.executable, '-m', 'pytest', '-q',
         'tests/test_m1_restart.py', 'tests/test_m2_restart.py', '-m', 'gpu'],
        cwd=PROJECT_ROOT, check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(f'CUDA smoke suite failed: exit={completed.returncode}')
    gpu_status = 'PASS'

M2_TEST_REPORT = {
    'accepted_m1_parent': 'PASS',
    'm0_m1_regression_cpu_suite': 'PASS',
    'm2_required_cpu_suite': 'PASS',
    'm2_fresh_process_resume': 'PASS',
    'optional_gpu_suite': gpu_status,
    'elapsed_s': time.monotonic() - test_started,
}
print(json.dumps(M2_TEST_REPORT, indent=2))


In [ ]:
import os

from src.m2_orchestrator import create_or_resume_m2
from src.runtime import environment_info, validate_persistent_root

PERSIST_ROOT = validate_persistent_root()
RUNTIME = environment_info()
print(json.dumps(RUNTIME, ensure_ascii=False, indent=2))
print('M2 config hash:', M2_CONFIG.config_hash())
print('Persistent root:', PERSIST_ROOT)

m2_orchestrator = create_or_resume_m2(
    PERSIST_ROOT,
    M2_CONFIG,
    PROJECT_ROOT,
    run_id=os.environ.get('VALIDATED_RG_M2_RUN_ID'),
    test_report=M2_TEST_REPORT,
)
print('再開用: export VALIDATED_RG_M2_RUN_ID=' + m2_orchestrator.state.run_id)
assert m2_orchestrator.state.milestone == 'M2'
assert m2_orchestrator.state.certification_status == 'NOT_CERTIFIED'

M2_RESULT = m2_orchestrator.run_until_checkpoint()
assert M2_RESULT['certification_status'] == 'NOT_CERTIFIED'
M2_RESULT


In [ ]:
loaded = m2_orchestrator.checkpoints.load_latest(restore_rng=False)
if loaded is None:
    raise RuntimeError('有効な committed checkpoint がありません。')
inspection = {
    'run_id': loaded.state.run_id,
    'checkpoint': str(loaded.path),
    'checkpoint_index': loaded.state.checkpoint_index,
    'phase': loaded.state.phase,
    'certification_status': loaded.state.certification_status,
    'done': sum(item.status == 'done' for item in loaded.queue.items.values()),
    'pending': sum(item.status == 'pending' for item in loaded.queue.items.values()),
    'tensor_shards': sorted(loaded.tensors),
    'skipped_invalid': list(loaded.skipped_invalid),
}
print(json.dumps(inspection, ensure_ascii=False, indent=2))

report_path = m2_orchestrator.run_root / 'reports' / 'M2_report.json'
if report_path.is_file():
    report = read_json(report_path)
    human_summary = {
        '判定': 'M2_COMPLETE。独立レビュー後にのみ M3 着手を判断する。',
        'acceptance_gates': report['acceptance_gates'],
        'heuristic_results': report['heuristic_results'],
        'memory': report['memory'],
        'checkpoint': report['checkpoint'],
        'remaining_todos': report['remaining_todos'],
        'certification_status': report['certification_status'],
    }
    print(json.dumps(human_summary, ensure_ascii=False, indent=2))
else:
    print('M2 は未完了です。next_session_plan.md に従って同じ run ID を再開してください。')
